In [1]:
print 'name'

SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (2075733539.py, line 1)

In [2]:
print(age)

NameError: name 'age' is not defined

In [4]:
age = 25
print(age)

25


In [5]:
num = [1,2,3,4,5]

num[5]

IndexError: list index out of range

In [6]:
import maths

ModuleNotFoundError: No module named 'maths'

In [7]:
math.PI

NameError: name 'math' is not defined

In [8]:
import math

math.PI

AttributeError: module 'math' has no attribute 'PI'

In [9]:
math.pi

3.141592653589793

In [10]:
users = {'name': 'Arun', 'age':28,'number':8015656774 }

users['dob']

KeyError: 'dob'

In [11]:
users['name']

'Arun'

In [12]:
4+'3'

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [13]:
from math import power

ImportError: cannot import name 'power' from 'math' (/opt/anaconda3/lib/python3.14/lib-dynload/math.cpython-314-darwin.so)

In [14]:
from math import pow

In [15]:
pow(2,3)

8.0

In [16]:
1/0

ZeroDivisionError: division by zero

In [1]:
pwd

'/Users/arunsankarask/ASK Second Brain/Skills/Python Mentor/Python Practice/30 Days Of Python'

## Quick note before the next two

Nice run through the built-ins — 9 of the 10 triggered and read correctly, including actually resolving the `ImportError` (`from math import power` fails, `from math import pow` works) instead of stopping at the traceback. One missing: **`ValueError`** — try `int('abc')` below to see it, since it's the one you'll hit constantly once real user/file input is involved.

This covered the *recognise* half (and Day 15's own "try the examples" ask). The two below are the *handle* half — `try`/`except`/`else`/`finally` — which is the part that actually closes the ROADMAP item, so don't stop here.

In [5]:
try:
    int('abc')
except ValueError:
    print('NaN')

NaN


## Exercise 2 (intermediate) — `safe_divide`, full try/except/else/finally

Write `safe_divide(a, b)`:

- `try`: return `a / b`
- `except ZeroDivisionError`: print a friendly message, return `None`
- `except TypeError`: print a different friendly message (non-numeric input), return `None`
- `else`: (only runs when no exception fired) print `"divided cleanly"`
- `finally`: always print `"safe_divide finished"` — prove it runs on every path, including the error ones

**Done when:** three calls — `safe_divide(10, 2)`, `safe_divide(10, 0)`, `safe_divide(10, 'a')` — each show the right branch firing and `finally` printing every time.

In [6]:
def safe_divide(a, b):
    try:
        result = a / b              # only the risky line goes in try
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    except TypeError:
        print("Both inputs need to be numbers")
        return None
    else:
        print("divided cleanly")    # runs only if try finished with no error
        return result
    finally:
        print("safe_divide finished")   # runs on every path


print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, 'a'))


divided cleanly
safe_divide finished
5.0
Cannot divide by zero
safe_divide finished
None
Both inputs need to be numbers
safe_divide finished
None


## Exercise 3 (project-tied) — Library Management System persistence

This is the exercise flagged on the ROADMAP: give the Library Management System real JSON persistence, wrapped properly.

- `save_library(library, filename)` — dumps the book list to JSON. Wrap the file write in `try/except`; if it fails, don't silently swallow it — print what went wrong and **re-raise**.
- `load_library(filename)`:
  - `except FileNotFoundError` → return an empty list (first run, no file yet — this is not an error case).
  - `except json.JSONDecodeError` → the file exists but is empty/corrupt — raise a clear `ValueError` naming the file, don't let the cryptic `JSONDecodeError` leak out to the caller.
- Wire both into your `Library` class (or call them from `main()`) so books persist across runs.

Paste your `Library`/`Book` classes into a cell above this one if they're not already here, or `import` them from wherever you kept them.

**Done when:** books survive a save → restart-kernel → load round trip, a missing file doesn't crash, and a deliberately corrupted file raises your own readable `ValueError` instead of a raw `JSONDecodeError`.

In [2]:
import importlib.util
import json

path = "/Users/arunsankarask/PythonProject-2026/Mini Project/Library Managemnt/Library Management.py"
spec = importlib.util.spec_from_file_location("library_mgmt", path)
lm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lm)

Book = lm.Book


In [3]:
def save_library(books, filename):
    """Write a list of Book objects to a JSON file."""
    try:
        with open(filename, "w") as f:
            json.dump([b.to_dict() for b in books], f, indent=2)
    except OSError as e:
        print(f"Could not save library to {filename!r}: {e}")
        raise                                  # tell the caller it failed


def load_library(filename):
    """Read Book objects from a JSON file. Missing file -> empty list."""
    try:
        with open(filename, "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        return []                              # first run: not an error
    except json.JSONDecodeError as e:
        raise ValueError(f"{filename!r} is empty or not valid JSON") from e
    else:
        return [Book.from_dict(d) for d in data]


class Library(lm.Library):
    """Your original Library, with load/save using the safe functions."""

    def __init__(self, filename):
        self.filename = filename       # set BEFORE super().__init__, which calls load()
        super().__init__()

    def load(self):
        self.books = load_library(self.filename)

    def save(self):
        save_library(self.books, self.filename)


In [4]:
lib = Library("books_test.json")
print(lib.books)          # [] and no crash
lib.add_book("Python", "Arun", "123")
lib.add_book("Tamil", "Shaliny", "333")


[<library_mgmt.Book object at 0x10ddfa270>, <library_mgmt.Book object at 0x10e0a9bd0>]
A book with ISBN 123 already exists
A book with ISBN 333 already exists


In [5]:
lib = Library("books_test.json")
for b in lib.list_books():
    print(b)              # both books are back


"Python by Arun (ISBN: 123)" - Available
"Tamil by Shaliny (ISBN: 333)" - Available


In [6]:
with open("bad.json", "w") as f:
    f.write("{not json")

Library("bad.json")       # ValueError: 'bad.json' is empty or not valid JSON


ValueError: 'bad.json' is empty or not valid JSON